In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 读取数据
country_df = pd.read_csv('/home/mw/input/data_total74037403/国家角度数据集/国家度量.csv', encoding='utf-8')

feature_cols = [col for col in country_df.columns if col not in ['国家名称', '年份']]

# 缺失值插值
country_df[feature_cols] = country_df[feature_cols].interpolate(method='linear', limit_direction='both')

def critic_weight(data):
    scaler = MinMaxScaler(feature_range=(0.001, 1))
    data_std = scaler.fit_transform(data)
    std = np.std(data_std, axis=0, ddof=1)
    corr = np.corrcoef(data_std, rowvar=False)
    conflict = 1 - np.abs(corr)
    info = std * np.sum(conflict, axis=1)
    weights = info / np.sum(info)
    return weights

def calculate_scores(data, weights):
    scaler = MinMaxScaler(feature_range=(0.001, 1))
    data_std = scaler.fit_transform(data)
    scores = np.sum(data_std * weights, axis=1)
    return scores

def gini(array):
    array = np.sort(array)
    n = len(array)
    cum = np.cumsum(array)
    return (n + 1 - 2 * np.sum(cum) / cum[-1]) / n

# 逐年处理
results = []
for year in sorted(country_df['年份'].dropna().unique()):
    year_data = country_df[country_df['年份'] == year]
    data = year_data[feature_cols].values
    weights = critic_weight(data)
    scores = calculate_scores(data, weights)
    gini_coef = gini(scores)
    results.append({
        '年份': year,
        '权重': dict(zip(feature_cols, weights)),
        '得分': dict(zip(year_data['国家名称'], scores)),
        '基尼系数': gini_coef
    })

# 输出权重和基尼系数
weights_df = pd.DataFrame([{**{'年份': r['年份']}, **r['权重']} for r in results])
gini_df = pd.DataFrame([(r['年份'], r['基尼系数']) for r in results], columns=['年份', '基尼系数'])

print("各年CRITIC权重：")
print(weights_df.set_index('年份'))
print("\n各年基尼系数：")
print(gini_df.set_index('年份'))

# 各国每年得分
scores_list = []
for r in results:
    for country, score in r['得分'].items():
        scores_list.append({'年份': r['年份'], '国家名称': country, '得分': score})
scores_df = pd.DataFrame(scores_list)
print("\n各国每年得分：")
print(scores_df)

weights_df.to_csv('/home/mw/input/data_total74037403/国家角度数据集/国家权重.csv', index=False, encoding='utf-8-sig')
gini_df.to_csv('/home/mw/input/data_total74037403/国家角度数据集/国家基尼系数.csv', index=False, encoding='utf-8-sig')
scores_df.to_csv('/home/mw/input/data_total74037403/国家角度数据集/国家评分.csv', index=False, encoding='utf-8-sig')

各年CRITIC权重：
       固定宽带普及率    科技期刊文章   独角兽公司数量
年份                                
2016  0.567955  0.239240  0.192805
2017  0.563257  0.245343  0.191400
2018  0.580113  0.225132  0.194755
2019  0.563076  0.236897  0.200027
2020  0.547585  0.245361  0.207054
2021  0.526370  0.252446  0.221185
2022  0.524257  0.246448  0.229295

各年基尼系数：
          基尼系数
年份            
2016  0.295573
2017  0.294527
2018  0.286601
2019  0.285460
2020  0.277252
2021  0.268249
2022  0.258762

各国每年得分：
       年份   国家名称        得分
0    2016    阿根廷  0.212062
1    2016    爱尔兰  0.366090
2    2016   澳大利亚  0.413124
3    2016     巴西  0.196951
4    2016    比利时  0.483906
..    ...    ...       ...
268  2022    意大利  0.381993
269  2022     印度  0.094320
270  2022  印度尼西亚  0.047822
271  2022     英国  0.487761
272  2022     中国  0.717308

[273 rows x 3 columns]
